In [ ]:
# GPU matte pre-pass for the v2.2.1 garment cropper.
# Generate the notebook with:
#   jupytext --to notebook --output ../crop_gpu.ipynb crop_gpu_cells.py
#
# Why this notebook exists: BiRefNet_lite costs ~75s per image on a laptop CPU
# and ~0.3-1s on a T4. Only the network forward pass needs the GPU; every
# refinement stage in garment_crop.py is cheap and stays local.
#
# Contract with the local pipeline: this writes 8-bit grayscale PNGs named
# {stem}.png, sized to the SOURCE image's (h, w), into v2/runs/.cache/matte/.
# garment_crop.biref_matte() reads exactly that and skips inference. The
# preprocessing below mirrors biref_matte() line for line — any divergence
# silently changes results, so do not "improve" it here in isolation.

# v2.2.1 — GPU matte pre-pass

Computes BiRefNet subject mattes for the cropper on a GPU runtime, so the
laptop never pays the ~75s/image CPU cost.

**Runtime → Change runtime type → T4 GPU** before running.

Outputs a zip of 8-bit mattes that drops into `v2/runs/.cache/matte/` locally;
after that a full 13-reference crop re-run takes about 8 seconds.

In [ ]:
#  1 · Runtime check --------------------------------------------------------
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip() or
      "NO GPU — set Runtime > Change runtime type > T4 before continuing")

In [ ]:
#  2 · Dependencies ---------------------------------------------------------
# onnxruntime-gpu supplies the CUDA execution provider; opencv/pillow match the
# local pipeline's image handling.
# pillow-avif-plugin: one Testset2 reference is .avif, which OpenCV cannot read
!pip install -q onnxruntime-gpu opencv-python-headless pillow pillow-avif-plugin

In [ ]:
#  3 · Source images --------------------------------------------------------
# Every source is now committed, so a shallow clone is the entire transfer.
#
# CRITICAL — matte dimensions must match what the local pipeline reads, because
# the local cache validates on exact (h, w) and silently recomputes on a
# mismatch. test_set/ is read raw locally, so no resize here. Testset2 is read
# locally from PREPPED copies (max side 1536, per ts2_harness.prep) which are
# gitignored, so the same resize must be applied here or every Testset2 matte is
# wasted work.
import os
REPO = "/content/tryon_repo"
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/101011101/magichour_takehome.git {REPO}

# (directory, max_side or None) — None means the local pipeline reads it raw
SOURCES = [(f"{REPO}/test_set/people", None),
           (f"{REPO}/Testset2/people", 1536),
           (f"{REPO}/Testset2/clothes", 1536),
           ("/content/extra", None)]
EXTS = (".jpg", ".jpeg", ".png", ".webp", ".avif")

srcs = []
for d, cap in SOURCES:
    if not os.path.isdir(d):
        continue
    for f in sorted(os.listdir(d)):
        if f.lower().endswith(EXTS):
            srcs.append((os.path.join(d, f), cap))
print(f"{len(srcs)} source images")
for d, cap in SOURCES:
    n = sum(1 for p, _ in srcs if os.path.dirname(p) == d)
    if n:
        print(f"  {n:3d}  {d}  cap={cap}")

In [ ]:
#  4 · Model ----------------------------------------------------------------
# Same checkpoint and URL as the local pipeline; a different export would change
# the mattes and break comparability with references already cached locally.
BIREF_URL = ("https://huggingface.co/onnx-community/BiRefNet_lite-ONNX/"
             "resolve/main/onnx/model.onnx")
MODEL = "/content/BiRefNet_lite.onnx"
if not os.path.exists(MODEL):
    !wget -q --show-progress -O {MODEL} {BIREF_URL}
print(f"{os.path.getsize(MODEL) / 1e6:.0f} MB")

In [ ]:
#  5 · Matte pre-pass -------------------------------------------------------
# Mirrors garment_crop.biref_matte(): INTER_AREA to 1024, BGR->RGB, imagenet
# normalise, CHW, sigmoid, INTER_CUBIC back to source size, 8-bit PNG.
import time, numpy as np, cv2, onnxruntime as ort

BIREF_SIDE = 1024
BIREF_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
BIREF_STD = np.array([0.229, 0.224, 0.225], np.float32)
OUT = "/content/matte"
os.makedirs(OUT, exist_ok=True)

sess = ort.InferenceSession(MODEL, providers=["CUDAExecutionProvider",
                                              "CPUExecutionProvider"])
print("providers:", sess.get_providers())
assert "CUDAExecutionProvider" in sess.get_providers(), \
    "CUDA not active — check the runtime type, otherwise this is no faster than the laptop"
name = sess.get_inputs()[0].name

import PIL.Image
try:
    import pillow_avif  # noqa: F401  — registers the AVIF decoder with PIL
except ImportError:
    pass


def load_bgr(path, cap):
    """OpenCV first; PIL for formats it cannot decode (avif). `cap` replicates
    ts2_harness.prep — PIL thumbnail semantics, max side, LANCZOS — so the matte
    matches the dimensions the local pipeline will ask for."""
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        im = PIL.Image.open(path).convert("RGB")
        bgr = cv2.cvtColor(np.asarray(im), cv2.COLOR_RGB2BGR)
    if cap and max(bgr.shape[:2]) > cap:
        im = PIL.Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        im.thumbnail((cap, cap), PIL.Image.LANCZOS)
        bgr = cv2.cvtColor(np.asarray(im), cv2.COLOR_RGB2BGR)
    return bgr


t_all = time.time()
for i, (path, cap) in enumerate(srcs, 1):
    stem = os.path.splitext(os.path.basename(path))[0]
    dst = os.path.join(OUT, f"{stem}.png")
    if os.path.exists(dst):
        continue
    try:
        bgr = load_bgr(path, cap)
    except Exception as e:
        print(f"  SKIP unreadable {stem}: {str(e)[:60]}")
        continue
    h, w = bgr.shape[:2]
    rgb = cv2.cvtColor(cv2.resize(bgr, (BIREF_SIDE, BIREF_SIDE),
                                  interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
    x = (rgb.astype(np.float32) / 255.0 - BIREF_MEAN) / BIREF_STD
    x = np.ascontiguousarray(x.transpose(2, 0, 1)[None])
    t = time.time()
    y = sess.run(None, {name: x})[0][0, 0]
    dt = time.time() - t
    p = 1.0 / (1.0 + np.exp(-y.astype(np.float32)))
    p = np.clip(cv2.resize(p, (w, h), interpolation=cv2.INTER_CUBIC), 0.0, 1.0)
    cv2.imwrite(dst, (p * 255).astype(np.uint8))
    print(f"  {i:3d}/{len(srcs)}  {stem:24s} {w}x{h}  {dt:5.2f}s  "
          f"fg={float((p > 0.5).mean()):.3f}")
print(f"total {time.time() - t_all:.1f}s for {len(srcs)} images")

In [ ]:
#  6 · Sanity check ---------------------------------------------------------
# A matte that is nearly all foreground or nearly all background is a failure,
# not a subject. Surface those now rather than discovering them in the crops.
import glob
from PIL import Image
bad = []
for f in sorted(glob.glob(f"{OUT}/*.png")):
    a = np.asarray(Image.open(f)).astype(np.float32) / 255.0
    fg = float((a > 0.5).mean())
    if fg < 0.02 or fg > 0.95:
        bad.append((os.path.basename(f), round(fg, 3)))
print(f"{len(glob.glob(f'{OUT}/*.png'))} mattes, {len(bad)} suspicious")
for b in bad:
    print("  ", b)

In [ ]:
#  7 · Preview a few --------------------------------------------------------
import matplotlib.pyplot as plt
show = sorted(glob.glob(f"{OUT}/*.png"))[:6]
fig, ax = plt.subplots(2, len(show), figsize=(3 * len(show), 9))
for j, f in enumerate(show):
    stem = os.path.splitext(os.path.basename(f))[0]
    src = next(((p, c) for p, c in srcs
                if os.path.splitext(os.path.basename(p))[0] == stem), None)
    ax[0, j].imshow(cv2.cvtColor(load_bgr(*src), cv2.COLOR_BGR2RGB))
    ax[1, j].imshow(np.asarray(Image.open(f)), cmap="gray")
    ax[0, j].set_title(stem, fontsize=9)
    for r in (0, 1):
        ax[r, j].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
#  8 · Export ---------------------------------------------------------------
# Unzip into v2/runs/.cache/matte/ locally, then re-run garment_crop.py — it
# will find every matte cached and skip inference entirely.
!cd /content && zip -qr mattes.zip matte
print(f"{os.path.getsize('/content/mattes.zip') / 1e6:.1f} MB")
from google.colab import files
files.download("/content/mattes.zip")

In [ ]:
#  8b · Optional: save to Drive instead ------------------------------------
# Per execution_conventions.md the project's Drive area is
# "Side projects and shi"; adjust if the folder name differs.
# from google.colab import drive
# drive.mount("/content/drive")
# !cp /content/mattes.zip "/content/drive/MyDrive/Side projects and shi/"